# 05 — Area GNN Forecaster Training

This notebook trains the Area-level intelligence models to forecast ward-level traffic congestion using Graph Neural Networks. It acts as the critical bridge coordinating individual ward agents.

We evaluate two models:
1. **WardPressureGCN**: Static spatial graph convolution over ward boundaries.
2. **SpatioTemporalGCN (STGCN)**: Combines spatial GCN layers with temporal GRU layers over a moving window of ward features.

Training dataset `global_temporal_data.pt` is generated by `04_ward_training.ipynb`.

In [ ]:
import sys
from pathlib import Path
import torch
from IPython.display import Image, display

# Ensure project root is in path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from train_gnn import train_for_area
from src.topology import Topology

topology = Topology(PROJECT_ROOT)
available_areas = topology.get_all_area_ids()
print(f"Available Areas: {available_areas}")

## 1. Load the Temporal Dataset

Load the dataset collected during ward training. It contains sequential observations of all 16 training wards.

In [ ]:
dataset_path = PROJECT_ROOT / "models" / "gnn" / "global_temporal_data.pt"

if not dataset_path.exists():
    raise FileNotFoundError("Please run Notebook 04 first to generate the dataset.")

dataset = torch.load(dataset_path, weights_only=False)
print(f"Loaded {len(dataset)} temporal samples.")

## 2. Train GCN (Spatial Only)

First, we train a simple static GCN. We'll train specifically on **HSR_Layout** to demonstrate.

In [ ]:
AREA = "HSR_Layout"
EPOCHS = 100
GNN_DIR = PROJECT_ROOT / "models" / "gnn"
PLOT_DIR = PROJECT_ROOT / "results" / "training"

print(f"Training GCN on {AREA}...")
gcn_result = train_for_area(
    area_id=AREA,
    topology=topology,
    data_path=dataset_path,
    gnn_dir=GNN_DIR,
    epochs=EPOCHS,
    model_type="gcn",
    plot_dir=PLOT_DIR
)

In [ ]:
# Display GCN Loss
display(Image(filename=PLOT_DIR / f"area_gcn_{AREA}_loss.png"))

## 3. Train SpatioTemporalGCN (GCN + GRU)

Now we train the STGCN architecture. This model buffers sequences of ward features over time, runs a spatial convolution at each timestep, and feeds the sequence into a GRU to predict pressure.

In [ ]:
print(f"Training STGCN on {AREA}...")
stgcn_result = train_for_area(
    area_id=AREA,
    topology=topology,
    data_path=dataset_path,
    gnn_dir=GNN_DIR,
    epochs=EPOCHS,
    model_type="stgcn",
    plot_dir=PLOT_DIR
)

In [ ]:
# Display STGCN Loss
display(Image(filename=PLOT_DIR / f"area_stgcn_{AREA}_loss.png"))

## 4. Compare Architectures

We can directly compare the predictive capability (MSE loss) of both architectures.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(gcn_result["losses"], label="GCN", linewidth=1.5, color="#6366f1")
ax1.plot(stgcn_result["losses"], label="STGCN", linewidth=1.5, color="#f43f5e")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE Loss")
ax1.set_title(f"Training Loss Comparison — {AREA}")
ax1.legend()
ax1.grid(alpha=0.3)

names = ["GCN", "STGCN"]
mses = [gcn_result["final_mse"], stgcn_result["final_mse"]]
bars = ax2.bar(names, mses, color=["#6366f1", "#f43f5e"], alpha=0.85)
for bar, mse in zip(bars, mses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{mse:.5f}", ha="center", va="bottom", fontsize=10)
ax2.set_ylabel("Final MSE")
ax2.set_title("Final Convergence MSE")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()